# 05. Colibrì Pure-C Engine Compilation & Warm Staging

Compiles the Colibrì inference engine from source with architecture-native vector optimizations and prepares local NVMe warm staging (~12.2 GiB) from Google Drive (`/content/drive/MyDrive/AI - Google Drive/GLM-5.2/model`).

In [ ]:
# 1. Bootstrap repository in Colab runtime
import os, sys, subprocess
REPO_DIR = '/content/glm52-drive-runtime'
if not os.path.exists(REPO_DIR):
    print(f"Cloning GLM-5.2 repository into {REPO_DIR}...")
    subprocess.run(['git', 'clone', 'https://github.com/Aqib2607/AI.git', REPO_DIR], check=True)

# 2. Install build toolchain
!apt-get update -qq && apt-get install -y -qq build-essential libgomp1 git

# 3. Clone and build Colibri source
!rm -rf /content/colibri
!git clone https://github.com/JustVugg/colibri.git /content/colibri

# 4. Build optimized binary with native vector math and OpenMP
%cd /content/colibri
!make glm ARCH=native
!./coli doctor
%cd /content

In [ ]:
# 5. Warm Staging to Colab Local Fast NVMe (~12.2 GiB)
import os
import shutil

DRIVE_MODEL_DIR = '/content/drive/MyDrive/AI - Google Drive/GLM-5.2/model'
LOCAL_MODEL_DIR = '/content/model'
os.makedirs(LOCAL_MODEL_DIR, exist_ok=True)

WARM_FILES = [
    'config.json',
    'generation_config.json',
    'tokenizer_config.json',
    'tokenizer.json',
    'out-mtp-00000.safetensors',
    'out-00000.safetensors'
]

print(f"Staging warm subset ({len(WARM_FILES)} files, ~12.2 GiB) to {LOCAL_MODEL_DIR}...")
for fname in WARM_FILES:
    src = os.path.join(DRIVE_MODEL_DIR, fname)
    dst = os.path.join(LOCAL_MODEL_DIR, fname)
    if os.path.exists(src):
        if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(src):
            print(f"  Copying {fname} ({os.path.getsize(src):,} bytes)...")
            shutil.copy2(src, dst)
        else:
            print(f"  ✓ {fname} already staged")
    else:
        print(f"  ! Warning: {src} not found in Google Drive")

print("\n✓ Warm local staging complete!")